# Custom Exception Handling in Python

Goal: design and use custom exception classes — hierarchies, extra attributes, error codes, chaining, and validation patterns.

## 1. Why Custom Exceptions?

Reusing generic exceptions (`ValueError`, `Exception`) everywhere makes it hard for callers to distinguish *why* something failed without parsing message strings. Custom exceptions let you:

- Give errors a **name** that documents intent (`OrderNotFoundError` vs `ValueError`).
- Attach **structured data** (ids, codes, field names) instead of only a message.
- Let callers catch broad or narrow categories via a **class hierarchy**.

The only requirement: a custom exception must (directly or indirectly) subclass `BaseException` — in practice, always subclass `Exception`.

In [1]:
class AppError(Exception):
    """Base class for all errors raised by our application."""


try:
    raise AppError("something went wrong")
except Exception as e:
    print(f"{type(e).__name__}: {e}")
    print(f"is subclass of Exception: {isinstance(e, Exception)}")

AppError: something went wrong
is subclass of Exception: True


## 2. Building an Exception Hierarchy

Give your app a single **base exception**, then subclass it per failure category. Callers can catch the base class to handle "any app error" broadly, or a specific subclass to handle one case precisely.

In [2]:
class AppError(Exception):
    """Base class for all application errors."""


class DataError(AppError):
    """Problems with data (not found, invalid, etc.)."""


class NotFoundError(DataError):
    def __init__(self, resource: str, identifier):
        self.resource = resource
        self.identifier = identifier
        super().__init__(f"{resource} with id={identifier!r} not found")


class ValidationError(DataError):
    def __init__(self, field: str, reason: str):
        self.field = field
        self.reason = reason
        super().__init__(f"Invalid value for '{field}': {reason}")


class AuthError(AppError):
    """Problems with authentication/authorization."""


class PermissionDeniedError(AuthError):
    def __init__(self, action: str):
        self.action = action
        super().__init__(f"Permission denied for action: {action}")


users = {1: {"name": "Alice", "role": "admin"}, 2: {"name": "Bob", "role": "viewer"}}


def get_user(user_id):
    if user_id not in users:
        raise NotFoundError("user", user_id)
    return users[user_id]


def require_admin(user_id):
    user = get_user(user_id)
    if user["role"] != "admin":
        raise PermissionDeniedError("delete_account")
    return user


for uid in (1, 2, 99):
    try:
        print(f"user {uid}: {require_admin(uid)}")
    except DataError as e:
        # catches NotFoundError and ValidationError specifically
        print(f"Data problem: {e}")
    except AuthError as e:
        # catches PermissionDeniedError specifically
        print(f"Auth problem: {e}")
    except AppError as e:
        # fallback for any other app error
        print(f"App error: {e}")

user 1: {'name': 'Alice', 'role': 'admin'}
Auth problem: Permission denied for action: delete_account
Data problem: user with id=99 not found


## 3. Custom Exceptions with Error Codes

Attaching a stable **error code** (and optionally other structured details) to a custom exception lets calling code — an API layer, logger, or monitoring system — branch on the error programmatically instead of parsing the message string. A common pattern:

- Define an error-code enum.
- Give the base exception a `code` attribute, defaulting per subclass.
- Add a `to_dict()` for serializing into API responses/logs.

In [3]:
from enum import Enum


class ErrorCode(Enum):
    UNKNOWN = "E000"
    NOT_FOUND = "E404"
    VALIDATION_FAILED = "E422"
    INSUFFICIENT_FUNDS = "E409"


class CodedAppError(Exception):
    """Base class for application errors that carry a machine-readable code."""
    code = ErrorCode.UNKNOWN

    def __init__(self, message: str, code: ErrorCode = None, **details):
        super().__init__(message)
        self.message = message
        self.code = code or self.code
        self.details = details

    def to_dict(self):
        return {
            "error_code": self.code.value,
            "message": self.message,
            "details": self.details,
        }

    def __str__(self):
        return f"[{self.code.value}] {self.message}"


class CodedNotFoundError(CodedAppError):
    code = ErrorCode.NOT_FOUND


class CodedValidationError(CodedAppError):
    code = ErrorCode.VALIDATION_FAILED


class InsufficientFundsError(CodedAppError):
    code = ErrorCode.INSUFFICIENT_FUNDS

    def __init__(self, balance, amount):
        super().__init__(
            f"Cannot withdraw {amount}, balance is only {balance}",
            balance=balance,
            amount=amount,
        )


def withdraw(balance, amount):
    if not isinstance(amount, (int, float)):
        raise CodedValidationError("amount must be a number", field="amount", got=type(amount).__name__)
    if amount > balance:
        raise InsufficientFundsError(balance, amount)
    return balance - amount


for balance, amount in [(100, 40), (100, 250), (100, "50")]:
    try:
        print(f"New balance: {withdraw(balance, amount)}")
    except CodedAppError as e:
        # Dispatch on the error code, similar to branching on an HTTP status
        if e.code is ErrorCode.INSUFFICIENT_FUNDS:
            print(f"Handled as funds issue -> {e.to_dict()}")
        else:
            print(f"Handled generically -> {e.to_dict()}")

New balance: 60
Handled as funds issue -> {'error_code': 'E409', 'message': 'Cannot withdraw 250, balance is only 100', 'details': {'balance': 100, 'amount': 250}}
Handled generically -> {'error_code': 'E422', 'message': 'amount must be a number', 'details': {'field': 'amount', 'got': 'str'}}


## 4. Chaining a Custom Exception onto a Lower-Level One

When a custom exception is raised in response to catching another exception, use `raise ... from ...` to preserve the original as `__cause__` instead of losing it.

In [4]:
class ConfigError(AppError):
    def __init__(self, key: str, reason: str):
        self.key = key
        self.reason = reason
        super().__init__(f"Invalid config for '{key}': {reason}")


def load_timeout(config: dict):
    try:
        return int(config["timeout"])
    except KeyError as e:
        raise ConfigError("timeout", "missing") from e
    except ValueError as e:
        raise ConfigError("timeout", "must be an integer") from e


try:
    load_timeout({"timeout": "not-a-number"})
except ConfigError as e:
    print(f"{e}")
    print(f"Caused by: {type(e.__cause__).__name__}: {e.__cause__}")

Invalid config for 'timeout': must be an integer
Caused by: ValueError: invalid literal for int() with base 10: 'not-a-number'


## 5. Collecting Multiple Errors in One Custom Exception

Sometimes you want to report *all* problems at once (e.g. form validation) rather than stopping at the first one. A custom exception can hold a list of sub-errors.

In [5]:
class FormValidationError(AppError):
    def __init__(self, errors: dict):
        self.errors = errors  # {field: reason}
        summary = ", ".join(f"{f}: {r}" for f, r in errors.items())
        super().__init__(f"{len(errors)} validation error(s) -> {summary}")


def validate_signup(form: dict):
    errors = {}
    if not form.get("email") or "@" not in form["email"]:
        errors["email"] = "must be a valid email"
    if len(form.get("pin", "")) < 8:
        errors["pin"] = "must be at least 8 characters"
    if not form.get("age") or form["age"] < 13:
        errors["age"] = "must be 13 or older"
    if errors:
        raise FormValidationError(errors)
    return "signup accepted"


bad_form = {"email": "not-an-email", "pin": "", "age": 10}
try:
    validate_signup(bad_form)
except FormValidationError as e:
    print(e)
    for field, reason in e.errors.items():
        print(f"  - {field}: {reason}")

3 validation error(s) -> email: must be a valid email, pin: must be at least 8 characters, age: must be 13 or older
  - email: must be a valid email
  - pin: must be at least 8 characters
  - age: must be 13 or older
